In [ ]:
from probill.agents import TeachableMcpAgent, McpHostAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_ext.tools.mcp import StdioServerParams, mcp_server_tools
from autogen_agentchat.ui import Console
from typing import Dict
from autogen_ext.experimental.task_centric_memory.utils import ChatCompletionClientRecorder
import json
import os
from probill.utils import create_oai_client, load_yaml_file, create_stdio_server, export_component

config_filepath = "./test.yaml"
config = load_yaml_file(config_filepath)

os.environ["OBSIDIAN_API_KEY"] = config["Obsidian"]["api_key"]

client = create_oai_client(config["client"])

server_params=create_stdio_server(config["StdioServerParams"])

mode = "replay" #replay

client = ChatCompletionClientRecorder(
    client=client,
    mode=mode,
    session_file_path="./sessions/session.json",
)

In [ ]:
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

async def run():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(
            read, write
        ) as session:
            # Initialize the connection
            await session.initialize()
            print("session initiated", flush=True)
            # List available prompts
            prompts = await session.list_prompts()

            # Get a prompt
            # prompt = await session.get_prompt(
            #     "evaluation", arguments={"patient_id": "P010", "start_page_id": "BINV-20", "clinical_data": {"HER2": "positive"}}
            # )

            # List available resources
            resources = await session.list_resources()

            # List available tools
            tools = await session.list_tools()
            for tool in tools.tools:
                print(tool.name)
                print(tool.description)
            # # Read a resource
            # content, mime_type = await session.read_resource("file://some/path")

            # Call a tool
            # result = await session.call_tool(
            #     "evaluate_patient_guidelines", 
            #     arguments={
            #         "patient_id": "P010",
            #         "start_page_id": "BINV-20"
            #     }
            # )
            
            # print(result.content)

await run()

In [ ]:
agent = McpHostAgent(
    name="obsidian_agent",
    model_client=client,
    description="Agent interacts with Obsidian contents",
    server_params=server_params,
    model_client_stream=True,
    system_message="You are an agent uses MCP tools to fufill the user's request. Say TERMINATE when you done.",
    reflect_on_tool_use=True,
)

In [ ]:
task = """List documents"""
result = await Console(agent.run_stream(task=task), output_stats=True)